# PR Governance Agent - Live Demo

**Fully self-contained.** Just fill in your configuration below, then click **Runtime > Run all**.
The notebook will automatically create a branch, open a PR, trigger the agent, and show results.

> **Security**: All URLs and tokens are masked and handled through inputs. No secrets are stored in this file.

## Step 1: Configure Environment

In [ ]:
from getpass import getpass
import urllib.request, json, time, re

# --- 1. Target Repository ---
REPO = input("GitHub Repository (e.g., owner/repo): ") or "lmudu2/risk-analyzer-poc"

# --- 2. Live Agent Endpoint ---
GATEWAY_URL = getpass("Live Agent Gateway URL: ")

# --- 3. Jira & Email Config (for display) ---
JIRA_DOMAIN = input("Jira Domain (e.g., name.atlassian.net): ") or "lmudu95.atlassian.net"
APPROVER_EMAIL = input("Approver Email (for display): ") or "lmudu95@gmail.com"

# --- 4. GitHub Authentication ---
GITHUB_TOKEN = getpass("GitHub PAT (repo scope): ")

GH_HEADERS = {
    "Authorization": f"token {GITHUB_TOKEN}",
    "Accept": "application/vnd.github.v3+json",
    "Content-Type": "application/json",
    "User-Agent": "PR-Agent-Demo"
}

def gh_get(path):
    url = f"https://api.github.com/repos/{REPO}/{path}"
    req = urllib.request.Request(url, headers=GH_HEADERS)
    with urllib.request.urlopen(req, timeout=10) as r:
        return json.loads(r.read())

def gh_post(path, data):
    url = f"https://api.github.com/repos/{REPO}/{path}"
    req = urllib.request.Request(url, data=json.dumps(data).encode(), headers=GH_HEADERS, method='POST')
    with urllib.request.urlopen(req, timeout=15) as r:
        return json.loads(r.read())

print(f"\n✅ Target Configured: {REPO}")
print(f"✅ Endpoint Masked: {GATEWAY_URL[:10]}...{GATEWAY_URL[-10:] if len(GATEWAY_URL)>20 else ''}")
print("Ready!")

## Step 2: Auto-Create a Test PR

In [ ]:
# Get the default branch SHA
repo_info = gh_get("")
default_branch = repo_info["default_branch"]
branch_info = gh_get(f"git/refs/heads/{default_branch}")
base_sha = branch_info["object"]["sha"]

# Create a new branch
branch_name = f"demo-agent-test-{int(time.time())}"
gh_post("git/refs", {"ref": f"refs/heads/{branch_name}", "sha": base_sha})

# Add a test file
import base64
file_content = base64.b64encode(f"# Demo test file\nCreated at {time.strftime('%Y-%m-%d %H:%M:%S')}\n".encode()).decode()
gh_post(f"contents/demo_test_{int(time.time())}.md", {
    "message": "demo: add test file",
    "content": file_content,
    "branch": branch_name
})

# Open the PR
pr = gh_post("pulls", {
    "title": f"[DEMO] Agent Security Audit - {time.strftime('%H:%M:%S')}",
    "head": branch_name,
    "base": default_branch,
    "body": "This PR was auto-created by the demo notebook."
})
PR_NUMBER = str(pr["number"])
PR_URL = pr["html_url"]
print(f"PR #{PR_NUMBER} created and ready for analysis.")

## Step 3: Trigger High Risk Analysis

In [ ]:
high_risk = {
    "action": "opened",
    "pull_request": {
        "number": int(PR_NUMBER),
        "title": "Refactor: delete primary identity column",
        "user": {"login": "demo-presenter", "type": "User"}
    },
    "repository": {"full_name": REPO},
    "installation": {"id": 12345678},
    "sender": {"login": "demo-presenter", "type": "User"}
}
req = urllib.request.Request(GATEWAY_URL, data=json.dumps(high_risk).encode(),
    headers={'Content-Type': 'application/json', 'X-GitHub-Event': 'pull_request'}, method='POST')
with urllib.request.urlopen(req, timeout=30) as res:
    print(f"Trigger sent successfully (HTTP {res.getcode()}). Analysis in progress...")

## Step 4: View Live Results

In [ ]:
print("Waiting 20 seconds for processing...")
time.sleep(20)

pr_data = gh_get(f"pulls/{PR_NUMBER}")
state = pr_data.get('state','?').upper()
merged = pr_data.get('merged', False)
icon = 'MERGED' if merged else ('CLOSED - Blocked by Agent' if state=='CLOSED' else 'OPEN')
print(f"\n=== PR STATUS ===")
print(f"PR #{PR_NUMBER}: {icon}")

comments = gh_get(f"issues/{PR_NUMBER}/comments")
jira = None
print(f"\n=== AI RISK REPORT ===")
for c in reversed(comments or []):
    body = c.get('body','')
    if 'RISK' in body.upper():
        print(body[:800] + "...")
        m = re.search(r'([A-Z]+-\d+)', body)
        if m: jira = m.group(1)
        break
else:
    print("No report yet - please wait and re-run.")

print(f"\n=== JIRA TICKET ===")
if jira:
    print(f"Created: {jira}")
    print(f"URL: https://{JIRA_DOMAIN}/browse/{jira}")
else:
    print("No ticket created for this risk level.")

print(f"\n=== EMAIL NOTIFICATION ===")
if state == 'CLOSED' and not merged:
    print(f"Approval email sent to {APPROVER_EMAIL[:3]}...@{APPROVER_EMAIL.split('@')[-1]}")
else:
    print("No email required for this flow.")